# Fine-tune YOLO for general face detection

Trains on WIDER FACE (converted to YOLO) + `fddb_dataset_YOLO`, both prepared locally by `scripts/prepare_wider_face.py`.

**Before running:** zip your project's `datasets/` folder (just the `WIDER_yolo/`, `fddb_dataset_YOLO/`, and `face_dataset.yaml` pieces are enough) and upload it to Google Drive, e.g. as `MyDrive/knowing-eye-datasets.zip`.

**Runtime:** Runtime -> Change runtime type -> T4 GPU (free tier is fine for a nano/small model).

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip the dataset you uploaded to Drive into the local Colab disk (much faster I/O than reading from Drive directly)
DATASET_ZIP = '/content/drive/MyDrive/knowing-eye-datasets.zip'
DATA_ROOT = '/content/datasets'

!mkdir -p {DATA_ROOT}
!unzip -q -o "{DATASET_ZIP}" -d {DATA_ROOT}
!find {DATA_ROOT} -maxdepth 2

In [ ]:
# Rewrite face_dataset.yaml's `path` to point at the extracted location in this Colab runtime.
# (The local copy has `path` pointing at your Windows machine, which won't exist here.)
import yaml

yaml_path = f'{DATA_ROOT}/face_dataset.yaml'
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = DATA_ROOT
with open(yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f)
print(cfg)

In [ ]:
from ultralytics import YOLO

# yolov8n = nano, smallest/fastest - good fit for a real-time webcam proctoring pipeline.
# Swap for 'yolov8s.pt' if you want more accuracy and can afford the extra latency.
model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # early stop if val mAP plateaus
    project='/content/runs',
    name='face_yolov8n',
    single_cls=True,      # only one class: face
)

In [ ]:
# Validate on the held-out val split
metrics = model.val()
print(metrics.box.map, metrics.box.map50)

In [ ]:
# Quick sanity check: run inference on a few val images and view predictions
import glob
sample_images = glob.glob(f'{DATA_ROOT}/WIDER_yolo/images/val/*.jpg')[:6]
preds = model.predict(sample_images, save=True, conf=0.4)
for p in preds:
    print(p.save_dir)

In [ ]:
# Copy the trained weights back to Drive so they survive the Colab session ending
import shutil, os

best_pt = '/content/runs/face_yolov8n/weights/best.pt'
dest_dir = '/content/drive/MyDrive/knowing-eye-weights'
os.makedirs(dest_dir, exist_ok=True)
shutil.copy2(best_pt, dest_dir)
print(f'Saved to {dest_dir}/best.pt')

In [ ]:
# Optional: export to ONNX for faster / non-torch inference in the backend pipeline
model.export(format='onnx')
!cp /content/runs/face_yolov8n/weights/best.onnx {dest_dir}/